# Step 0: Pre-flight Check
This cell ensures a few things:
- The Python environment is configured
- `nebius` CLI is installed, with a valid profile
- `.env` is populated correctly

The cell below loads `.env`, checks that required entries are present, and reads the bucket with the provided keys.

**NO GPU jobs will be run**.

In [4]:
from dotenv import load_dotenv
load_dotenv(".env")

import nebius_job as nj
import notebook_utils as nu

ready = nu.local_check()
print("\nReady to launch!" if ready else "\nfix the FAILs above (see docs/NEBIUS_SETUP.md)")

[PASS] nebius CLI on PATH: /home/jd/.nebius/bin/nebius
[PASS] CLI profile can list jobs
[PASS] SDG_BUCKET set
[PASS] SDG_S3_ENDPOINT set
[PASS] SDG_S3_REGION set
[PASS] NEBIUS_S3_ACCESS_KEY_ID set
[PASS] NEBIUS_S3_SECRET_ACCESS_KEY set
[PASS] bucket reachable with .env keys: sdg-image-bucket

Ready to launch!


# Step 1: Serverless Job Launch (Dry Run)

In [8]:
nj.launch("configs/naive_20.json", dry_run=True)

==> naive_20: First attempt at stock-example density: a handful of objects, so most frames are near-empty. The gate should catch it.
nebius \
  ai \
  job \
  create \
  --name \
  sdg-naive-20-20261018-195250 \
  --image \
  nvcr.io/nvidia/isaac-sim@sha256:c2f47dc82a7714af08d3766efe80ac9d084c2b37b5d0dfbd074797ec56390fc7 \
  --platform \
  gpu-l40s-a \
  --preset \
  1gpu-8vcpu-32gb \
  --disk-size \
  100Gi \
  --timeout \
  1h \
  --restart-policy \
  never \
  --container-command \
  bash \
  --args \
  '/workspace/nebius/job_entry.sh naive_20.json' \
  --working-dir \
  /workspace \
  --inject-file \
  run_sdg.py:/workspace/run_sdg.py \
  --inject-file \
  constants.py:/workspace/constants.py \
  --inject-file \
  sdg/generate_replicator.py:/workspace/sdg/generate_replicator.py \
  --inject-file \
  sdg/check_validation.py:/workspace/sdg/check_validation.py \
  --inject-file \
  nebius/job_entry.sh:/workspace/nebius/job_entry.sh \
  --inject-file \
  configs/common.json:/workspace/

# Step 2: Probe Nebius

In [10]:
probe_id = nj.launch("configs/probe.json")
probe = nj.wait(probe_id)
nj.logs(probe_id, tail=40)
print("\n", probe["state"], probe["durations_s"])

==> probe: No generator: checks GPU, Vulkan, network and bucket access from inside a job, then renders one frame.
nebius \
  ai \
  job \
  create \
  --name \
  sdg-probe-20261018-200633 \
  --image \
  nvcr.io/nvidia/isaac-sim@sha256:c2f47dc82a7714af08d3766efe80ac9d084c2b37b5d0dfbd074797ec56390fc7 \
  --platform \
  gpu-rtx6000 \
  --preset \
  1gpu-24vcpu-218gb \
  --disk-size \
  100Gi \
  --timeout \
  1h \
  --restart-policy \
  never \
  --container-command \
  bash \
  --args \
  '/workspace/nebius/job_entry.sh --probe' \
  --working-dir \
  /workspace \
  --inject-file \
  run_sdg.py:/workspace/run_sdg.py \
  --inject-file \
  constants.py:/workspace/constants.py \
  --inject-file \
  sdg/generate_replicator.py:/workspace/sdg/generate_replicator.py \
  --inject-file \
  sdg/check_validation.py:/workspace/sdg/check_validation.py \
  --inject-file \
  nebius/job_entry.sh:/workspace/nebius/job_entry.sh \
  --inject-file \
  configs/common.json:/workspace/configs/common.json \
  -

RuntimeError: nebius ai job create failed (3):
Error: rpc error: code = InvalidArgument desc = plaintext must not exceed 65536 bytes
request = 2cd74d6d-ba3a-43ab-8eb1-6d29775a909f
caused by service error:
  Bad Request BadRequest: service kms-dpl, violations:
    plaintext: plaintext must not exceed 65536 bytes

This issue may be linked to API request calls. The following information may be helpful while debugging the issue:
  Trace ID: 17bb96e407469f8b907df32a5803f70c
  Previous Request IDs: c254a5b1-a1b6-4ba0-b905-5f002b649b4b, e8fb7ead-a3e4-4833-b843-f3aa443851ae